# Mental Health Text Classification Analysis


## 1. Setup and Data Loading

This section handles the necessary library imports and loads the dataset into a pandas DataFrame for initial inspection and processing.

In [1]:
# Import standard libraries for data handling and system operations
import numpy as np 
import pandas as pd 
import os
import re
import time

# Import NLTK for text processing
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab', quiet=True) # Download tokenizer data

# Import scikit-learn for feature extraction, model selection, and classification
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# Import XGBoost for gradient boosting classification
from xgboost import XGBClassifier

# Import PyTorch for deep learning models
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from collections import Counter

print('All necessary libraries imported successfully.')

All necessary libraries imported successfully.


### Loading the Dataset


In [8]:
# List files in the input directory to confirm data source
for dirname, _, filenames in os.walk('/Datasets'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Load the dataset into a pandas DataFrame
df = pd.read_csv(r"Datasets\mental_heath_unbanlanced.csv")

### Initial Data Inspection


In [9]:
# Display the first 5 rows of the DataFrame
print("DataFrame Head:")
print(df.head())

# Display 5 random rows of the DataFrame
print("\nDataFrame Sample:")
print(df.sample(5))

DataFrame Head:
   Unique_ID                                               text   status
0        0.0                                         oh my gosh  Anxiety
1        1.0  trouble sleeping, confused mind, restless hear...  Anxiety
2        2.0  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3        3.0  I've shifted my focus to something else but I'...  Anxiety
4        4.0  I'm restless and restless, it's been a month n...  Anxiety

DataFrame Sample:
       Unique_ID                                               text  \
8314      8985.0  4000 miles from nowhere...loneliness is overwh...   
5499      5694.0                                      jyp se muri0?   
32517    39522.0  every time i talk to somebody outside of my ho...   
12971    14948.0  I have been planning my suicide for a while. T...   
35324    42523.0  nicolerichie yes we had the vhs i cried when t...   

           status  
8314     Suicidal  
5499       Normal  
32517  Depression  
12971    Suicidal  


The `Unique_ID` column appears to be an identifier and is not relevant for the classification task. We will remove it to simplify the DataFrame.

In [10]:
# Drop the 'Unique_ID' column
df = df.drop(columns='Unique_ID')

### Target Variable Analysis and Encoding

The target variable `status` contains categorical labels. We inspect its distribution and then convert these labels into numerical representations for model training.

In [11]:
# Display the distribution of the 'status' column
print("Distribution of Mental Health Status:")
print(df.status.value_counts())

# Create a mapping from string labels to integer codes
status_map = {
    'Anxiety': 0,
    'Depression': 1,
    'Normal': 2,
    'Suicidal': 3
}

# Apply the mapping to create a new numerical target column 'sts_int'
df['sts_int'] = df['status'].map(status_map)

print("\nUpdated DataFrame Head with encoded status:")
print(df.head())

Distribution of Mental Health Status:
status
Normal        18391
Depression    14506
Suicidal      11212
Anxiety        5503
Name: count, dtype: int64

Updated DataFrame Head with encoded status:
                                                text   status  sts_int
0                                         oh my gosh  Anxiety        0
1  trouble sleeping, confused mind, restless hear...  Anxiety        0
2  All wrong, back off dear, forward doubt. Stay ...  Anxiety        0
3  I've shifted my focus to something else but I'...  Anxiety        0
4  I'm restless and restless, it's been a month n...  Anxiety        0


## 2. Text Preprocessing

Text data often contains noise, inconsistent formatting, and irrelevant characters that can hinder model performance. This section defines functions to clean and tokenize the raw text, preparing it for feature extraction.

### Text Cleaning Function

The `clean_text` function performs several crucial cleaning steps:

*   **Reduces Repeated Characters**: E.g., "hellooooo" becomes "helloo".
*   **Replaces Numbers**: All numeric sequences are replaced with a `NUM` token to generalize digits.
*   **ASCII Conversion**: Non-ASCII characters are ignored to handle encoding issues.
*   **URL Replacement**: Web links are replaced with a `LINK` token.

In [12]:
def clean_text(text):
    """Applies various cleaning steps to a given text string."""
    # Replace repeated characters with at most two occurrences (e.g., 'nooooo' -> 'noo')
    text = re.sub(r'(.)\1+', r'\1\1', text)

    # Replace all numbers with a generic 'NUM' token
    text = re.sub(r'(\d)+', ' NUM ', text)

    # Encode to ASCII and decode, ignoring non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Replace URLs with a generic 'LINK' token
    url_pattern = r'https?://\S+|www\.\S+'
    text = re.sub(url_pattern, ' LINK ', text)

    return text

Let's test the `clean_text` function with an example to see its effect.

In [13]:
# Example text to test the cleaning function
example_text = 'nooooooooooooo999999999ooo88898888-898898frhhg___________________. hi https://www.example.com hiiidamn I had a dream that I had a cute cute cat named PojiðŸ˜'
cleaned_example = clean_text(example_text)
print(f"Original text: {example_text}")
print(f"Cleaned text: {cleaned_example}")

Original text: nooooooooooooo999999999ooo88898888-898898frhhg___________________. hi https://www.example.com hiiidamn I had a dream that I had a cute cute cat named PojiðŸ˜
Cleaned text: noo NUM oo NUM - NUM frhhg__. hi  LINK  hiidamn I had a dream that I had a cute cute cat named Poji


### Custom Tokenizer Function

The `custom_tokenizer` function further processes the cleaned text by:

*   **Lowercasing**: Converts all text to lowercase.
*   **Word Tokenization**: Breaks down the text into individual words.
*   **Punctuation Removal**: Removes leading/trailing punctuation and underscores from tokens.

In [14]:
def custom_tokenizer(text):
    """Applies cleaning, lowercasing, and tokenization, then cleans tokens."""
    # Convert text to lowercase
    text = text.lower()
    # Apply the general cleaning function
    cleaned = clean_text(text)
    # Tokenize the cleaned text
    tokens = word_tokenize(cleaned)

    # Remove leading/trailing punctuation and underscores from each token
    cleaned_tokens = [re.sub(r'^[\W_]+|[\W_]+$', '', token) for token in tokens]

    # Filter out any empty tokens that might result from cleaning
    cleaned_tokens = [tok for tok in cleaned_tokens if tok]

    return cleaned_tokens

Let's test the `custom_tokenizer` function with the same example text.

In [15]:
# Test the custom tokenizer with the example text
tokenizer_test_result = custom_tokenizer(example_text)
print(f"Tokenized example: {tokenizer_test_result}")

Tokenized example: ['noo', 'NUM', 'oo', 'NUM', 'NUM', 'frhhg', 'hi', 'LINK', 'hiidamn', 'i', 'had', 'a', 'dream', 'that', 'i', 'had', 'a', 'cute', 'cute', 'cat', 'named', 'poji']


## 3. Feature Engineering and Classical Machine Learning (CountVectorizer)

This section explores using `CountVectorizer` to transform text into numerical features. `CountVectorizer` creates a bag-of-words representation, where each document is represented as a vector of word counts. We then train and evaluate several classical machine learning models on these features.

### Applying CountVectorizer

We initialize `CountVectorizer` with our `custom_tokenizer` and `stop_words='english'` to remove common English words. The vectorizer then transforms the text corpus into a sparse matrix of token counts.

In [16]:
# Initialize CountVectorizer with the custom tokenizer and English stop words
count_vectorizer = CountVectorizer(tokenizer=custom_tokenizer, stop_words='english')

# Fit and transform the text column from the DataFrame
corpus = df.text.tolist()
X_cv = count_vectorizer.fit_transform(corpus)

print("CountVectorizer Tokens (first 50): ", ', '.join(count_vectorizer.get_feature_names_out()[:50]))
print("Vocabulary Size (CountVectorizer):", len(count_vectorizer.get_feature_names_out()))

c:\Users\hayth\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


CountVectorizer Tokens (first 50):  LINK, NUM, a'olha, a'shit, a'ways, a+e, a-decade, a-hole, a-level, a-lot, a-okay, a-student, a.d.d, a.i, a.k.a, a.m, a.n.d, a.p, a/c, a_smt, aa, aa'inni, aa/na, aafter, aagaffufkccxkxnmdfuvkcidjdjjxnxkzkzknzkzkzlkzzkzkkznxkcicjkxkxbxnxkxmxnjzizmzmfukcicjnafuckyoukillyourselfcuntcuntcunthissmndyydjsnshxgvxbsjsjshxkkillme, aagainand, aaguthu, aagzhwhdb, aah, aahaahhahahahahhahahahhawhat, aahahhaahahahaa, aahh, aahhaa, aahmddr, aajata, aajtakkrishnakant, aak, aakhri, aakk, aalexaanne, aaliyah, aalsmeer, aamelia, aaminn, aan, aand, aandjxjsjjxjjcskkekckkcksk, aang, aanndd, aanyway
Vocabulary Size (CountVectorizer): 57109


Even after cleaning and removing stop words, the vocabulary size is quite large. This can lead to high-dimensional feature spaces, which might be computationally expensive and suffer from sparsity, especially for larger datasets.

### Model Training and Evaluation (CountVectorizer)

We split the data into training and testing sets, ensuring a stratified split to maintain class proportions. Three classifiers are then trained and evaluated:

*   **Logistic Regression**: A linear model often used as a baseline.
*   **Random Forest**: An ensemble method that can capture complex interactions.
*   **XGBoost**: A powerful gradient boosting algorithm known for high performance.

In [17]:
y = df['sts_int']

# Split data into train and test sets (70% train, 30% test) with stratification
X_train_cv, X_test_cv, y_train, y_test = train_test_split(X_cv, y, test_size=0.3, random_state=42, stratify=y)

# Define the models to be evaluated
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, multi_class='ovr', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
}

# Iterate through models, train, predict, and evaluate
for name, model in models.items():
    start_time = time.time()
    model.fit(X_train_cv, y_train)
    train_time = time.time() - start_time

    start_time = time.time()
    y_pred = model.predict(X_test_cv)
    pred_time = time.time() - start_time

    f1 = f1_score(y_test, y_pred, average='macro')

    print(f"--- {name} (CountVectorizer) ---")
    print(f"Train time: {train_time:.4f} sec")
    print(f"Prediction time: {pred_time:.4f} sec")
    print(f"Macro F1 score: {f1:.4f}")
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("\n")

c:\Users\hayth\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


--- Logistic Regression (CountVectorizer) ---
Train time: 11.6068 sec
Prediction time: 0.0079 sec
Macro F1 score: 0.7500
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.74      0.77      1651
           1       0.72      0.66      0.69      4352
           2       0.84      0.95      0.89      5517
           3       0.67      0.63      0.65      3364

    accuracy                           0.77     14884
   macro avg       0.76      0.74      0.75     14884
weighted avg       0.76      0.77      0.76     14884



--- Random Forest (CountVectorizer) ---
Train time: 322.4278 sec
Prediction time: 1.7166 sec
Macro F1 score: 0.6855
Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.45      0.60      1651
           1       0.62      0.74      0.68      4352
           2       0.81      0.93      0.87      5517
           3       0.68      0.53      0.60      3364

    ac

c:\Users\hayth\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:09:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- XGBoost (CountVectorizer) ---
Train time: 18.8854 sec
Prediction time: 0.4184 sec
Macro F1 score: 0.7573
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.73      0.78      1651
           1       0.72      0.69      0.70      4352
           2       0.84      0.93      0.88      5517
           3       0.69      0.63      0.66      3364

    accuracy                           0.77     14884
   macro avg       0.77      0.75      0.76     14884
weighted avg       0.77      0.77      0.77     14884





### Analysis of CountVectorizer Results

The results show that while 'Normal' status is classified relatively well, the performance for 'Anxiety', 'Depression', and 'Suicidal' categories is notably lower. This suggests the models struggle to distinguish between these categories, possibly due to semantic similarities in the language used to describe these states.

Key limitations observed with CountVectorizer:

*   **Lack of Semantic Understanding**: CountVectorizer only considers word frequency, not the meaning or context of words. Words like "stressed" might appear in both 'Anxiety' and 'Depression' texts, leading to confusion.
*   **High Dimensionality and Sparsity**: The large vocabulary leads to a high-dimensional feature space, which is predominantly sparse (most word counts are zero). This increases computational requirements and can sometimes hinder model performance.
*   **Suboptimal Accuracy**: The current F1 scores for critical categories are not satisfactory for real-world applications requiring nuanced classification.

## 4. Feature Engineering and Classical Machine Learning (TF-IDF Vectorizer)

To address some limitations of `CountVectorizer`, we next employ `TfidfVectorizer` (Term Frequency-Inverse Document Frequency). TF-IDF not only considers how often a word appears in a document (Term Frequency) but also how unique or important that word is across the entire corpus (Inverse Document Frequency). This weighting scheme often provides a more balanced representation, giving less weight to very common words like "the" and higher weight to more discriminative words.

### Applying TF-IDF Vectorizer and Model Evaluation

Similar to `CountVectorizer`, we use `TfidfVectorizer` with our `custom_tokenizer` and then train the same set of classical machine learning models. We then compare their performance.

In [18]:
# Initialize TfidfVectorizer with the custom tokenizer and English stop words
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=custom_tokenizer,
    stop_words='english',
    max_features=None  # Use all features, can be limited for larger datasets
)

# Fit and transform the text corpus
corpus = df.text.tolist()
X_tfidf = tfidf_vectorizer.fit_transform(corpus)

print("TF-IDF Vocabulary Size:", len(tfidf_vectorizer.get_feature_names_out()))

# Prepare target variable
y = df['sts_int']

# Split data into train and test sets with stratification
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Define the models (same as before)
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, multi_class='ovr', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
}

# Iterate through models, train, predict, and evaluate
for name, model in models.items():
    start_time = time.time()
    model.fit(X_train_tfidf, y_train)
    train_time = time.time() - start_time

    start_time = time.time()
    y_pred = model.predict(X_test_tfidf)
    pred_time = time.time() - start_time

    f1 = f1_score(y_test, y_pred, average='macro')

    print(f"--- {name} (TF-IDF) ---")
    print(f"Train time: {train_time:.4f} sec")
    print(f"Prediction time: {pred_time:.4f} sec")
    print(f"Macro F1 score: {f1:.4f}")
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("\n")

c:\Users\hayth\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


TF-IDF Vocabulary Size: 57109


c:\Users\hayth\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


--- Logistic Regression (TF-IDF) ---
Train time: 2.3483 sec
Prediction time: 0.0035 sec
Macro F1 score: 0.7514
Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.71      0.77      1651
           1       0.72      0.67      0.70      4352
           2       0.82      0.95      0.88      5517
           3       0.69      0.63      0.66      3364

    accuracy                           0.77     14884
   macro avg       0.77      0.74      0.75     14884
weighted avg       0.77      0.77      0.76     14884



--- Random Forest (TF-IDF) ---
Train time: 295.8131 sec
Prediction time: 1.7491 sec
Macro F1 score: 0.6883
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.47      0.62      1651
           1       0.61      0.75      0.67      4352
           2       0.82      0.94      0.88      5517
           3       0.70      0.50      0.58      3364

    accuracy             

c:\Users\hayth\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:16:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- XGBoost (TF-IDF) ---
Train time: 79.7617 sec
Prediction time: 0.4000 sec
Macro F1 score: 0.7614
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.75      0.79      1651
           1       0.71      0.70      0.70      4352
           2       0.85      0.94      0.89      5517
           3       0.69      0.63      0.66      3364

    accuracy                           0.78     14884
   macro avg       0.77      0.75      0.76     14884
weighted avg       0.77      0.78      0.77     14884





### Analysis of TF-IDF Results

We observe a slight improvement in performance compared to `CountVectorizer`, particularly in the F1 scores for some of the challenging categories. This indicates that incorporating word importance (rarity) across documents helps differentiate terms more effectively.

However, the fundamental limitations remain:

*   **Still No Semantic Understanding**: Both CountVectorizer and TF-IDF operate at the lexical level. They treat words as independent entities and do not capture contextual meaning, synonyms, or relationships between words (e.g., "happy" vs. "joyful"). This is crucial for distinguishing nuanced emotional states.
*   **High Sparsity**: Despite TF-IDF's weighting, the feature vectors are still high-dimensional and sparse, leading to similar computational challenges as with CountVectorizer.

For tasks requiring a deeper understanding of text content, especially subtle emotional cues, more advanced techniques that can capture semantic information are necessary. This naturally leads us to explore word embeddings and deep learning models like Recurrent Neural Networks.

## 5. Deep Learning Models for Text Classification

Traditional Bag-of-Words (BoW) methods like CountVectorizer and TF-IDF struggle with capturing the sequential nature and semantic meaning of text. Deep learning models, particularly Recurrent Neural Networks (RNNs) and their variants (like GRUs), are designed to process sequential data and learn rich, contextual embeddings for words. This section implements various RNN-based models to tackle the classification task.

### 5.1 Data Preparation for Deep Learning

For deep learning models, text data needs to be converted into numerical sequences of word indices. This involves:

*   **Tokenization**: Breaking text into words (reusing our `custom_tokenizer`).
*   **Vocabulary Building**: Creating a mapping from words to unique integers.
*   **Sequence Encoding**: Converting tokenized texts into sequences of integers.
*   **Padding**: Ensuring all sequences have the same length by adding special padding tokens.
*   **PyTorch Dataset and DataLoader**: Structuring the data for efficient batch processing during training.

In [19]:
# Get texts and labels from the DataFrame
texts = df['text'].tolist()
labels = df['sts_int'].tolist()

# Tokenize all texts using the custom tokenizer
tokenized_texts = [custom_tokenizer(t) for t in texts]

# Build vocabulary: Count word frequencies and create a set of unique words
word_counts = Counter(word for doc in tokenized_texts for word in doc)
vocab = {word for word, count in word_counts.items() if count >= 1} # Include all words appearing at least once

# Create word-to-index mapping, including special tokens for padding and unknown words
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word in vocab:
    word2idx[word] = len(word2idx)

# Encode tokenized texts into sequences of numerical indices
def encode_text(tokenized):
    return [word2idx.get(word, word2idx['<UNK>']) for word in tokenized]

encoded_texts = [encode_text(doc) for doc in tokenized_texts]

# Determine the maximum sequence length for padding
max_len = max(len(seq) for seq in encoded_texts) if encoded_texts else 0

def pad_seq(seq, max_len=max_len):
    """Pads a sequence to a fixed maximum length."""
    return seq + [word2idx['<PAD>']] * (max_len - len(seq))

# Pad all encoded texts to the maximum length
padded_texts = [pad_seq(seq) for seq in encoded_texts]

print(f"Vocabulary size: {len(word2idx)}")
print(f"Maximum sequence length: {max_len}")
print(f"Example of encoded and padded text: {padded_texts[0][:10]}...")

Vocabulary size: 57419
Maximum sequence length: 9684
Example of encoded and padded text: [28450, 19745, 50706, 0, 0, 0, 0, 0, 0, 0]...


In [20]:
# Define a custom PyTorch Dataset for text data
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Convert lists to PyTorch tensors
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

# Split the data into training and testing sets for deep learning
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    padded_texts, labels, test_size=0.3, random_state=42, stratify=labels
)

# Create Dataset objects
train_dataset = TextDataset(X_train_dl, y_train_dl)
test_dataset = TextDataset(X_test_dl, y_test_dl)

# Create DataLoader objects for batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

Number of training batches: 1086
Number of testing batches: 466


### 5.2 Advanced Bidirectional RNN Model

To further enhance sequence understanding, we introduce a more advanced RNN model with:

*   **Bidirectional Processing**: The RNN processes sequences in both forward and backward directions, capturing context from both past and future tokens.
*   **Dropout**: A regularization technique to prevent overfitting by randomly dropping units during training.
*   **Packed Padded Sequences**: An optimization for handling variable-length sequences efficiently in PyTorch RNNs, preventing unnecessary computation on padding tokens.

To correctly use `pack_padded_sequence`, the `DataLoader` needs to provide the actual lengths of the sequences before padding. We define a `collate_fn` to handle this.

In [21]:
# Custom collate_fn to pad sequences and return their original lengths
def collate_fn(batch):
    sequences = [item[0] for item in batch] # Extract sequences
    labels = torch.tensor([item[1] for item in batch]) # Extract labels
    
    # Calculate actual lengths of non-padding tokens
    # Ensure lengths are integers and reflect true sequence content
    lengths = torch.tensor([(seq != word2idx['<PAD>']).sum().item() for seq in sequences], dtype=torch.long)
    
    # Pad sequences to the longest sequence in the batch
    sequences_padded = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=word2idx['<PAD>'])
    
    return sequences_padded, lengths, labels

# Filter out any empty sequences that might arise from very aggressive tokenization and cleaning.
# A sequence with length 0 would cause issues with pack_padded_sequence.
filtered_train_data = [(seq, label) for seq, label in train_dataset if (seq != word2idx['<PAD>']).sum() > 0]
filtered_test_data = [(seq, label) for seq, label in test_dataset if (seq != word2idx['<PAD>']).sum() > 0]

# Update DataLoaders to use the custom collate_fn
train_loader_advanced = DataLoader(filtered_train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader_advanced = DataLoader(filtered_test_data, batch_size=32, collate_fn=collate_fn)

print("DataLoaders updated for handling sequence lengths.")

DataLoaders updated for handling sequence lengths.


In [22]:
class AdvancedRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, dropout=0.3):
        super(AdvancedRNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2idx['<PAD>'])
        self.dropout = nn.Dropout(dropout) # Dropout for regularization
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True # Bidirectional RNN to process sequences in both directions
        )
        # Output layer, hidden_dim * 2 because of bidirectional RNN (concatenating forward and backward states)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x, lengths):
        embedded = self.dropout(self.embedding(x)) # Apply dropout to embeddings
        
        # Pack padded sequence for efficient RNN processing
        # `enforce_sorted=False` allows lengths to not be pre-sorted, but might be slower.
        packed_embedded = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        
        # Pass packed sequence through RNN
        packed_output, hidden = self.rnn(packed_embedded)
        
        # For a bidirectional RNN, the last hidden state will have shape 
        # (num_layers * num_directions, batch, hidden_dim).
        # We concatenate the last forward (hidden[-2]) and last backward (hidden[-1]) hidden states.
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1) # hidden: (batch, hidden_dim * 2)
        out = self.fc(hidden)
        return out

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vocab_size = len(word2idx)
embedding_dim = 100
hidden_dim = 128
output_dim = len(set(labels))

# Instantiate Advanced RNN model
model = AdvancedRNNClassifier(vocab_size, embedding_dim, hidden_dim, output_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 6
print("Starting Advanced Bidirectional RNN training...")

# Training loop using the advanced DataLoader
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for texts_batch, lengths_batch, labels_batch in train_loader_advanced:
        texts_batch, lengths_batch, labels_batch = texts_batch.to(device), lengths_batch.to(device), labels_batch.to(device)
        optimizer.zero_grad()
        outputs = model(texts_batch, lengths_batch) # Pass lengths to the model
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader_advanced):.4f}")

print("Advanced Bidirectional RNN training complete.")

Starting Advanced Bidirectional RNN training...
Epoch 1/6 - Loss: 1.1484
Epoch 2/6 - Loss: 1.0319
Epoch 3/6 - Loss: 1.0477
Epoch 4/6 - Loss: 1.0339
Epoch 5/6 - Loss: 1.0313
Epoch 6/6 - Loss: 1.0452
Advanced Bidirectional RNN training complete.


In [25]:
print(torch.cuda.is_available())

False


### Evaluate Advanced Bidirectional RNN Model

We evaluate the `AdvancedRNNClassifier` model's performance on the test set.

In [26]:
# Set model to evaluation mode
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for texts_batch, lengths_batch, labels_batch in test_loader_advanced:
        texts_batch, lengths_batch, labels_batch = texts_batch.to(device), lengths_batch.to(device), labels_batch.to(device)
        outputs = model(texts_batch, lengths_batch)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

print("Advanced Bidirectional RNN Classification Report:")
print(classification_report(all_labels, all_preds))

macro_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"Advanced Bidirectional RNN Macro F1 Score: {macro_f1:.4f}")

Advanced Bidirectional RNN Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.24      0.35      1651
           1       0.45      0.61      0.52      4352
           2       0.70      0.76      0.73      5516
           3       0.46      0.33      0.38      3364

    accuracy                           0.56     14883
   macro avg       0.57      0.49      0.50     14883
weighted avg       0.57      0.56      0.55     14883

Advanced Bidirectional RNN Macro F1 Score: 0.4963


### 5.3 Advanced Bidirectional GRU Model

Combining the strengths of GRUs (better gradient flow) with bidirectional processing and dropout, we create an `AdvancedGRUClassifier`. This model is expected to offer robust performance by understanding context from both directions while efficiently handling long-term dependencies.

In [27]:
class AdvancedGRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, dropout=0.3):
        super(AdvancedGRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2idx['<PAD>'])
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x, lengths):
        embedded = self.dropout(self.embedding(x))
        
        # Pack padded sequence for GRU
        packed_embedded = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.gru(packed_embedded)
        
        # Concatenate final forward and backward hidden states
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.fc(hidden)
        return out

In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vocab_size = len(word2idx)
embedding_dim = 100
hidden_dim = 128
output_dim = len(set(labels))

# Instantiate Advanced GRU model
model = AdvancedGRUClassifier(vocab_size, embedding_dim, hidden_dim, output_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 6
print("Starting Advanced Bidirectional GRU training...")

# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for texts_batch, lengths_batch, labels_batch in train_loader_advanced:
        texts_batch, lengths_batch, labels_batch = texts_batch.to(device), lengths_batch.to(device), labels_batch.to(device)
        optimizer.zero_grad()
        outputs = model(texts_batch, lengths_batch)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader_advanced):.4f}")

print("Advanced Bidirectional GRU training complete.")

Starting Advanced Bidirectional GRU training...
Epoch 1/6 - Loss: 0.8437
Epoch 2/6 - Loss: 0.5718
Epoch 3/6 - Loss: 0.4990
Epoch 4/6 - Loss: 0.4467
Epoch 5/6 - Loss: 0.4029
Epoch 6/6 - Loss: 0.3629
Advanced Bidirectional GRU training complete.


### Evaluate Advanced Bidirectional GRU Model

We evaluate the `AdvancedGRUClassifier` model's performance on the test set.

In [29]:
# Set model to evaluation mode
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for texts_batch, lengths_batch, labels_batch in test_loader_advanced:
        texts_batch, lengths_batch, labels_batch = texts_batch.to(device), lengths_batch.to(device), labels_batch.to(device)
        outputs = model(texts_batch, lengths_batch)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

print("Advanced Bidirectional GRU Classification Report:")
print(classification_report(all_labels, all_preds))

macro_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"Advanced Bidirectional GRU Macro F1 Score: {macro_f1:.4f}")

Advanced Bidirectional GRU Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.80      0.81      1651
           1       0.67      0.79      0.73      4352
           2       0.93      0.91      0.92      5516
           3       0.72      0.59      0.64      3364

    accuracy                           0.79     14883
   macro avg       0.79      0.77      0.78     14883
weighted avg       0.79      0.79      0.79     14883

Advanced Bidirectional GRU Macro F1 Score: 0.7754


In [31]:
import torch, json, os

os.makedirs("gru_api", exist_ok=True)

# 1. Save model weights
torch.save(model.state_dict(), "gru_api/gru_model.pth")

# 2. Save vocabulary
with open("gru_api/word2idx.json", "w") as f:
    json.dump(word2idx, f)

# 3. Save config
model_config = {
    "vocab_size": vocab_size,
    "embedding_dim": embedding_dim,
    "hidden_dim": hidden_dim,
    "output_dim": output_dim,
    "max_len": len(padded_texts[0])  # max sequence length
}
with open("gru_api/config.json", "w") as f:
    json.dump(model_config, f)

print("All files saved to gru_api/")

All files saved to gru_api/


In [32]:
print(df[['status', 'sts_int']].drop_duplicates().sort_values('sts_int'))

          status  sts_int
0        Anxiety        0
6734  Depression        1
727       Normal        2
6737    Suicidal        3
